# FPGA Neural Network v5 — 3 Fixed Sensors Training Pipeline

**Architecture:** 3 → 8 (ReLU) → 3 → argmax  
**Quantization:** Q6.10 signed fixed-point (16-bit)  
**Classes:** 0=FORWARD, 1=STOP, 2=TURN  
**Sensors:** 3 fixed HC-SR04 (no servo scan)  

### Labeling Rules (v5)
- **FORWARD (0):** center ≥ 40 AND left ≥ 25 AND right ≥ 25
- **STOP (1):** center < 15 AND left < 15 AND right < 15 (truly cornered)
- **TURN (2):** center < 40 OR left < 25 OR right < 25 (but not all cornered)

### Key Techniques
- Deduplication of stuck/repetitive readings (>3cm change threshold)
- Left↔Right mirroring to eliminate directional bias
- Synthetic data for under-represented classes (STOP, side approaches)
- L2 weight decay (0.0005) to keep weights within Q6.10 range
- FPGA bit-accurate simulation before deployment

## 1. Load & Clean Data

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load data3.txt (3 fixed sensor data)
DATA_FILE = "../arduino/de1soc_data_collect/data3.txt"

lines = []
with open(DATA_FILE) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("=") or line.startswith("Speed") or line.startswith("Flip") or "dist_left" in line:
            continue
        parts = line.split(",")
        if len(parts) == 7:
            try:
                lines.append([int(p) for p in parts])
            except ValueError:
                continue

data = np.array(lines)
print(f"Total rows parsed: {len(data)}")

# Remove class=-1 (FPGA not responding)
valid = data[data[:, 6] != -1]
print(f"After removing class=-1: {len(valid)}")

# Deduplicate: remove rows where L,C,R barely changed from previous
# This removes the ~400 rows where the robot was stuck at a wall
kept = [0]
for i in range(1, len(valid)):
    prev = valid[kept[-1]]
    curr = valid[i]
    if abs(curr[0]-prev[0]) > 3 or abs(curr[1]-prev[1]) > 3 or abs(curr[2]-prev[2]) > 3:
        kept.append(i)

deduped = valid[kept]
print(f"After deduplication (>3cm change): {len(deduped)} (removed {len(valid)-len(deduped)} repetitive rows)")

dist_left   = deduped[:, 0].astype(float)
dist_center = deduped[:, 1].astype(float)
dist_right  = deduped[:, 2].astype(float)

print(f"\nDistance ranges:")
print(f"  Left:   {dist_left.min():.0f} - {dist_left.max():.0f} cm")
print(f"  Center: {dist_center.min():.0f} - {dist_center.max():.0f} cm")
print(f"  Right:  {dist_right.min():.0f} - {dist_right.max():.0f} cm")

## 2. Apply Labeling Rules

In [ ]:
old_labels = deduped[:, 6].copy()
new_labels = np.zeros(len(deduped), dtype=int)

for i in range(len(deduped)):
    L, C, R = dist_left[i], dist_center[i], dist_right[i]
    
    # STOP (class 1): ALL three < 15 — truly cornered
    if C < 15 and L < 15 and R < 15:
        new_labels[i] = 1
    # TURN (class 2): center < 40 OR any side < 25
    elif C < 40 or L < 25 or R < 25:
        new_labels[i] = 2
    # FORWARD (class 0): all clear
    else:
        new_labels[i] = 0

changed = np.sum(old_labels != new_labels)
print(f"Labels changed: {changed} / {len(deduped)} ({100*changed/len(deduped):.1f}%)")
print()
for cls, name in [(0, "FORWARD"), (1, "STOP"), (2, "TURN")]:
    old_count = np.sum(old_labels == cls)
    new_count = np.sum(new_labels == cls)
    print(f"  {name}: {old_count} → {new_count}")

X_raw = np.column_stack([dist_left, dist_center, dist_right])
y = new_labels
X = X_raw / 400.0  # Normalize to [0, 1]

print(f"\nNormalized input range: [{X.min():.3f}, {X.max():.3f}]")

## 3. Explore Data Distribution

In [ ]:
print("=" * 50)
print("DATA DISTRIBUTION BY CLASS")
print("=" * 50)

for cls, name in [(0, "FORWARD"), (1, "STOP"), (2, "TURN")]:
    mask = y == cls
    count = np.sum(mask)
    if count == 0:
        print(f"\n{name}: 0 samples (will be added synthetically)")
        continue
    print(f"\n{name}: {count} samples")
    print(f"  Left:   min={X_raw[mask, 0].min():.0f}  max={X_raw[mask, 0].max():.0f}  mean={X_raw[mask, 0].mean():.0f}")
    print(f"  Center: min={X_raw[mask, 1].min():.0f}  max={X_raw[mask, 1].max():.0f}  mean={X_raw[mask, 1].mean():.0f}")
    print(f"  Right:  min={X_raw[mask, 2].min():.0f}  max={X_raw[mask, 2].max():.0f}  mean={X_raw[mask, 2].mean():.0f}")

# Show some example rows per class
print("\n" + "=" * 50)
print("SAMPLE ROWS PER CLASS")
print("=" * 50)
for cls, name in [(0, "FORWARD"), (1, "STOP"), (2, "TURN")]:
    mask = y == cls
    if np.sum(mask) == 0:
        continue
    indices = np.where(mask)[0][:5]
    print(f"\n{name}:")
    for idx in indices:
        print(f"  L={X_raw[idx,0]:.0f}  C={X_raw[idx,1]:.0f}  R={X_raw[idx,2]:.0f}")

## 4. Augment & Balance

In [ ]:
np.random.seed(42)
X_aug, y_aug = [X], [y]

# --- Noise augmentation (5 noisy copies) ---
for _ in range(5):
    noise = np.random.normal(0, 0.02, X.shape)
    X_aug.append(np.clip(X + noise, 0, 1))
    y_aug.append(y)

# --- Mirror Left↔Right (fixes directional bias) ---
# The robot mostly approached walls from the left in data collection,
# so mirroring ensures symmetric detection.
X_mirror = X[:, [2, 1, 0]]  # swap left and right columns
X_aug.append(X_mirror)
y_aug.append(y)
for _ in range(5):
    noise = np.random.normal(0, 0.02, X_mirror.shape)
    X_aug.append(np.clip(X_mirror + noise, 0, 1))
    y_aug.append(y)

print("Mirrored L↔R to fix directional bias")

# --- Synthetic STOP (all < 15cm) ---
for _ in range(400):
    L = np.random.uniform(0, 14) / 400
    C = np.random.uniform(0, 14) / 400
    R = np.random.uniform(0, 14) / 400
    X_aug.append(np.array([[L, C, R]]))
    y_aug.append(np.array([1]))

# --- Synthetic TURN: left side close ---
for _ in range(400):
    L = np.random.uniform(3, 24) / 400
    C = np.random.uniform(20, 200) / 400
    R = np.random.uniform(25, 400) / 400
    X_aug.append(np.array([[L, C, R]]))
    y_aug.append(np.array([2]))

# --- Synthetic TURN: right side close ---
for _ in range(400):
    L = np.random.uniform(25, 400) / 400
    C = np.random.uniform(20, 200) / 400
    R = np.random.uniform(3, 24) / 400
    X_aug.append(np.array([[L, C, R]]))
    y_aug.append(np.array([2]))

# --- Synthetic TURN: center close ---
for _ in range(400):
    L = np.random.uniform(15, 400) / 400
    C = np.random.uniform(5, 39) / 400
    R = np.random.uniform(15, 400) / 400
    X_aug.append(np.array([[L, C, R]]))
    y_aug.append(np.array([2]))

# --- Synthetic TURN: tight corners (not fully cornered) ---
for _ in range(300):
    L = np.random.uniform(15, 30) / 400
    C = np.random.uniform(15, 39) / 400
    R = np.random.uniform(15, 30) / 400
    X_aug.append(np.array([[L, C, R]]))
    y_aug.append(np.array([2]))

# --- Synthetic FORWARD: clearly safe ---
for _ in range(300):
    L = np.random.uniform(40, 400) / 400
    C = np.random.uniform(45, 400) / 400
    R = np.random.uniform(40, 400) / 400
    X_aug.append(np.array([[L, C, R]]))
    y_aug.append(np.array([0]))

X_all = np.vstack(X_aug)
y_all = np.concatenate(y_aug)

print(f"\nTotal after augmentation: {len(y_all)}")

# --- Balance classes by oversampling ---
counts = [np.sum(y_all == c) for c in range(3)]
max_count = max(counts)
print(f"Before balancing: FWD={counts[0]}, STOP={counts[1]}, TURN={counts[2]}")

X_bal, y_bal = [], []
for cls in range(3):
    mask = y_all == cls
    X_c, y_c = X_all[mask], y_all[mask]
    if len(y_c) < max_count:
        idx = np.random.choice(len(y_c), max_count - len(y_c), replace=True)
        X_c = np.vstack([X_c, X_c[idx]])
        y_c = np.concatenate([y_c, y_c[idx]])
    X_bal.append(X_c)
    y_bal.append(y_c)

X_train = np.vstack(X_bal).astype(np.float32)
y_train = np.concatenate(y_bal).astype(np.int64)

# Shuffle
perm = np.random.permutation(len(y_train))
X_train, y_train = X_train[perm], y_train[perm]

print(f"After balancing: {len(y_train)} total")
for cls, name in [(0, "FWD"), (1, "STOP"), (2, "TURN")]:
    print(f"  {name}: {np.sum(y_train == cls)}")

## 5. Train Neural Network

In [ ]:
# --- Activation functions ---
def relu(x):
    return np.maximum(0, x)

def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

# --- One-hot encode labels ---
y_onehot = np.zeros((len(y_train), 3), dtype=np.float32)
y_onehot[np.arange(len(y_train)), y_train] = 1.0

# --- Initialize weights ---
np.random.seed(42)
W1 = np.random.randn(3, 8).astype(np.float32) * 0.3
b1 = np.zeros(8, dtype=np.float32)
W2 = np.random.randn(8, 3).astype(np.float32) * 0.3
b2 = np.zeros(3, dtype=np.float32)

# --- Hyperparameters ---
learning_rate = 0.005
weight_decay = 0.0005   # L2 regularization to prevent Q6.10 overflow
epochs = 1500
batch_size = 32

best_acc = 0
best_W1 = best_b1 = best_W2 = best_b2 = None
history = []

print(f"Training: {len(X_train)} samples, {epochs} epochs, batch_size={batch_size}")
print(f"Learning rate: {learning_rate}, Weight decay: {weight_decay}")
print()

for epoch in range(epochs):
    # Shuffle each epoch
    perm = np.random.permutation(len(X_train))
    X_shuf, y_shuf = X_train[perm], y_onehot[perm]
    
    for i in range(0, len(X_shuf), batch_size):
        Xb = X_shuf[i:i+batch_size]
        yb = y_shuf[i:i+batch_size]
        
        # Forward pass
        z1 = Xb @ W1 + b1
        a1 = relu(z1)
        z2 = a1 @ W2 + b2
        a2 = softmax(z2)
        
        # Backward pass
        m = len(Xb)
        dz2 = (a2 - yb) / m
        dW2 = a1.T @ dz2 + weight_decay * W2
        db2 = dz2.sum(axis=0)
        dz1 = (dz2 @ W2.T) * (z1 > 0).astype(float)
        dW1 = Xb.T @ dz1 + weight_decay * W1
        db1 = dz1.sum(axis=0)
        
        # Update weights
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
    
    # Track accuracy every 100 epochs
    if (epoch + 1) % 100 == 0:
        z1 = X_train @ W1 + b1
        a1 = relu(z1)
        z2 = a1 @ W2 + b2
        preds = z2.argmax(axis=1)
        acc = (preds == y_train).mean()
        history.append((epoch + 1, acc))
        
        if acc > best_acc:
            best_acc = acc
            best_W1, best_b1 = W1.copy(), b1.copy()
            best_W2, best_b2 = W2.copy(), b2.copy()
        
        if (epoch + 1) % 200 == 0:
            print(f"  Epoch {epoch+1:5d}: acc={acc:.4f}  |W1|_max={np.abs(W1).max():.2f}  |W2|_max={np.abs(W2).max():.2f}")

# Use best weights
W1, b1 = best_W1, best_b1
W2, b2 = best_W2, best_b2

print(f"\nBest training accuracy: {best_acc:.4f}")
print(f"Max weight magnitude: W1={np.abs(W1).max():.2f}, W2={np.abs(W2).max():.2f}")
print(f"Total parameters: {W1.size + b1.size + W2.size + b2.size}")

## 6. Quantize to Q6.10 Fixed-Point

In [ ]:
def float_to_q610(val):
    """Convert float to Q6.10 signed 16-bit integer.
    Q6.10 = 1 sign bit + 5 integer bits + 10 fractional bits.
    Range: -32.0 to +31.999 (resolution: 1/1024 ≈ 0.001)
    """
    q = int(round(val * 1024))  # Multiply by 2^10
    q = max(-32768, min(32767, q))  # Clamp to 16-bit signed
    return q & 0xFFFF  # Return as unsigned 16-bit for Verilog

def q610_to_float(q):
    """Convert Q6.10 unsigned 16-bit back to float."""
    if q >= 32768:
        q -= 65536  # Sign extend
    return q / 1024.0

# Quantize all weights and biases
W1_q = np.vectorize(float_to_q610)(W1)
b1_q = np.vectorize(float_to_q610)(b1)
W2_q = np.vectorize(float_to_q610)(W2)
b2_q = np.vectorize(float_to_q610)(b2)

# Show quantization error
W1_recon = np.vectorize(q610_to_float)(W1_q)
W2_recon = np.vectorize(q610_to_float)(W2_q)

print("Quantization Summary:")
print(f"  W1 float range: [{W1.min():.4f}, {W1.max():.4f}]")
print(f"  W1 Q6.10 range: [0x{int(W1_q.min()):04X}, 0x{int(W1_q.max()):04X}]")
print(f"  W1 max quantization error: {np.abs(W1 - W1_recon).max():.6f}")
print(f"  W2 float range: [{W2.min():.4f}, {W2.max():.4f}]")
print(f"  W2 Q6.10 range: [0x{int(W2_q.min()):04X}, 0x{int(W2_q.max()):04X}]")
print(f"  W2 max quantization error: {np.abs(W2 - W2_recon).max():.6f}")

# Check for overflow risk
max_w = max(np.abs(W1).max(), np.abs(W2).max())
if max_w > 31.0:
    print(f"  ⚠ WARNING: max weight {max_w:.2f} exceeds Q6.10 range!")
else:
    print(f"  ✓ All weights within Q6.10 range (max={max_w:.2f}, limit=31.999)")

## 7. FPGA Bit-Accurate Simulation

In [ ]:
def fpga_inference(in0, in1, in2, W1_q, b1_q, W2_q, b2_q):
    """
    Simulates the exact Verilog state machine in neural_net.v.
    Inputs: Q6.10 values (byte * 4, where byte is 0-255)
    Arithmetic: 32-bit signed accumulator, result extracted from bits [25:10]
    """
    inputs = [in0, in1, in2]
    
    # Layer 1: 3 inputs × 8 neurons with ReLU
    l1 = []
    for neu in range(8):
        # Bias initialization: acc = bias * 1024 (shift to Q6.20)
        b = int(b1_q[neu])
        if b >= 32768: b -= 65536
        acc = b * 1024
        
        # Multiply-accumulate
        for inp in range(3):
            w = int(W1_q[inp, neu])
            if w >= 32768: w -= 65536
            acc += inputs[inp] * w
        
        # ReLU + extract bits [25:10]
        if acc < 0:
            l1.append(0)
        else:
            l1.append((acc >> 10) & 0xFFFF)
    
    # Layer 2: 8 inputs × 3 neurons (no ReLU)
    l2 = []
    for neu in range(3):
        b = int(b2_q[neu])
        if b >= 32768: b -= 65536
        acc = b * 1024
        
        for inp in range(8):
            w = int(W2_q[inp, neu])
            if w >= 32768: w -= 65536
            v = l1[inp]
            if v >= 32768: v -= 65536
            acc += v * w
        
        result = (acc >> 10) & 0xFFFF
        if result >= 32768: result -= 65536
        l2.append(result)
    
    # Argmax
    if l2[0] >= l2[1] and l2[0] >= l2[2]:
        return 0  # FORWARD
    elif l2[1] >= l2[2]:
        return 1  # STOP
    else:
        return 2  # TURN


# --- Test on original (non-augmented) data ---
correct = 0
confusion = np.zeros((3, 3), dtype=int)

for i in range(len(X_raw)):
    # Convert cm to byte (0-255) then to Q6.10 (byte * 4)
    bL = int(np.clip(X_raw[i, 0] * 255 / 400, 0, 255))
    bC = int(np.clip(X_raw[i, 1] * 255 / 400, 0, 255))
    bR = int(np.clip(X_raw[i, 2] * 255 / 400, 0, 255))
    
    pred = fpga_inference(bL * 4, bC * 4, bR * 4, W1_q, b1_q, W2_q, b2_q)
    
    if pred == y[i]:
        correct += 1
    confusion[y[i], pred] += 1

fpga_acc = correct / len(y)
print(f"FPGA Bit-Accurate Accuracy: {fpga_acc:.4f} ({correct}/{len(y)})")
print(f"\nConfusion Matrix (rows=true, cols=predicted):")
print(f"{'':>8} FWD   STOP  TURN")
for cls, name in [(0, "FWD  "), (1, "STOP "), (2, "TURN ")]:
    print(f"  {name}  {confusion[cls, 0]:5d} {confusion[cls, 1]:5d} {confusion[cls, 2]:5d}")

print(f"\nPer-Class Accuracy:")
for cls, name in [(0, "FORWARD"), (1, "STOP"), (2, "TURN")]:
    total = confusion[cls].sum()
    if total > 0:
        print(f"  {name}: {confusion[cls, cls]}/{total} = {100*confusion[cls, cls]/total:.1f}%")
    else:
        print(f"  {name}: no samples in test data")

# Safety check: how many times did the model say FORWARD when it should have said TURN?
fwd_when_turn = confusion[2, 0]
print(f"\n*** SAFETY: Model said FORWARD when should TURN: {fwd_when_turn} times ***")
if fwd_when_turn == 0:
    print("    → Robot will NEVER drive into an obstacle it should turn from.")

## 8. Critical Scenario Tests

In [ ]:
CLASS_NAMES = ["FWD", "STOP", "TURN"]

scenarios = [
    # (description, left_cm, center_cm, right_cm, expected_class)
    ("All clear (100, 100, 100)",     100, 100, 100, 0),
    ("All clear (200, 200, 200)",     200, 200, 200, 0),
    ("All max (400, 400, 400)",       400, 400, 400, 0),
    ("Center close (50, 30, 50)",      50,  30,  50, 2),
    ("Center close (60, 20, 60)",      60,  20,  60, 2),
    ("Left wall (15, 50, 100)",        15,  50, 100, 2),
    ("Right wall (100, 50, 15)",      100,  50,  15, 2),
    ("Left close (10, 60, 100)",       10,  60, 100, 2),
    ("Right close (100, 60, 10)",     100,  60,  10, 2),
    ("Approaching (40, 35, 40)",       40,  35,  40, 2),
    ("Corner tight (20, 20, 20)",      20,  20,  20, 2),
    ("Cornered (10, 10, 10)",          10,  10,  10, 1),
    ("Cornered (5, 5, 5)",              5,   5,   5, 1),
    ("Side scrape L (20, 80, 200)",    20,  80, 200, 2),
    ("Side scrape R (200, 80, 20)",   200,  80,  20, 2),
    ("Corridor (30, 100, 30)",         30, 100,  30, 2),
    ("Open ahead (40, 200, 40)",       40, 200,  40, 0),
    ("One side (10, 200, 200)",        10, 200, 200, 2),
    ("Borderline (25, 40, 25)",        25,  40,  25, 0),
    ("Just under (24, 40, 25)",        24,  40,  25, 2),
    ("Right danger (150, 80, 8)",     150,  80,   8, 2),
    ("Left danger (8, 80, 150)",        8,  80, 150, 2),
]

passed = 0
print(f"{'Scenario':<35} {'Result':<6} {'Predicted':<6} {'Expected':<6}")
print("-" * 60)
for name, L, C, R, expected in scenarios:
    bL = int(np.clip(L * 255 / 400, 0, 255))
    bC = int(np.clip(C * 255 / 400, 0, 255))
    bR = int(np.clip(R * 255 / 400, 0, 255))
    pred = fpga_inference(bL * 4, bC * 4, bR * 4, W1_q, b1_q, W2_q, b2_q)
    ok = "PASS" if pred == expected else "FAIL"
    if pred == expected:
        passed += 1
    print(f"  {name:<33} {ok:<6} {CLASS_NAMES[pred]:<6} {CLASS_NAMES[expected]:<6}")

print(f"\nPassed: {passed}/{len(scenarios)}")

## 9. Export Verilog Weights

In [ ]:
def to_hex16(val):
    """Format Q6.10 value as Verilog hex literal."""
    return f"16'sh{int(val):04X}"

print("// ============================================")
print("// Neural Network Weights v5 — Paste into neural_net.v")
print("// Architecture: 3 → 8 (ReLU) → 3 → argmax")
print("// Trained on 3-fixed-sensor data with L↔R mirroring")
print("// ============================================")

print("\n// Layer 1 weights W1[input][neuron] — 3×8 = 24 weights")
for inp in range(3):
    for neu in range(8):
        idx = inp * 8 + neu
        float_val = q610_to_float(int(W1_q[inp, neu]))
        print(f"                {idx:2d}: get_w1={to_hex16(W1_q[inp, neu])};  // {float_val:+.4f}")

print("\n// Layer 1 biases b1[8]")
for neu in range(8):
    float_val = q610_to_float(int(b1_q[neu]))
    print(f"                 {neu}: get_b1={to_hex16(b1_q[neu])};  // {float_val:+.4f}")

print("\n// Layer 2 weights W2[input][neuron] — 8×3 = 24 weights")
for inp in range(8):
    for neu in range(3):
        idx = inp * 3 + neu
        float_val = q610_to_float(int(W2_q[inp, neu]))
        print(f"                {idx:2d}: get_w2={to_hex16(W2_q[inp, neu])};  // {float_val:+.4f}")

print("\n// Layer 2 biases b2[3]")
for neu in range(3):
    float_val = q610_to_float(int(b2_q[neu]))
    print(f"                {neu}: get_b2={to_hex16(b2_q[neu])};  // {float_val:+.4f}")

## 10. Save Weights & Clean Data

In [ ]:
import os

# Save quantized and float weights
save_path = "nn_weights_v5.npz"
np.savez(save_path,
         W1=W1, b1=b1, W2=W2, b2=b2,
         W1_q=W1_q, b1_q=b1_q, W2_q=W2_q, b2_q=b2_q)
print(f"Weights saved to {save_path}")
print(f"  File size: {os.path.getsize(save_path)} bytes")

# Save cleaned data as CSV
csv_path = "3sensor_data_cleaned.csv"
header = "dist_left,dist_center,dist_right,class"
csv_data = np.column_stack([X_raw, y])
np.savetxt(csv_path, csv_data, delimiter=",", header=header, comments="", fmt="%d")
print(f"Cleaned data saved to {csv_path}")
print(f"  {len(y)} rows (deduplicated, relabeled)")

print("\n" + "=" * 50)
print("DEPLOYMENT CHECKLIST")
print("=" * 50)
print("1. Copy weights from Section 9 into neural_net.v")
print("2. Compile in Quartus Prime Lite")
print("3. Program FPGA with new .sof")
print("4. Upload de1soc_phase5_3sensor.ino to Arduino")
print("5. Flip SW[0] on FPGA and test")
print("\nDone!")